In [ ]:
import json
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import json
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving chatbot_intent_augmentation.json to chatbot_intent_augmentation.json


In [ ]:
with open("chatbot_intent_augmentation.json", "r", encoding="utf-8") as file:
    data = json.load(file)

In [ ]:
sentences = []
labels = []

In [ ]:
for item in data:
  sentences.append(item["text"])
  labels.append(item["intent"])

In [ ]:
vectorizer = TfidfVectorizer(lowercase = True, ngram_range=(1,2),sublinear_tf=True)
x = vectorizer.fit_transform(sentences)

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(labels)

In [ ]:
model = LogisticRegression(max_iter=20000,random_state=30)
model.fit(x,y)

LogisticRegression(max_iter=20000, random_state=30)

In [ ]:
with open("chatbot_model.pkl", "wb") as file:
    pickle.dump(model, file)

In [ ]:
with open("vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

In [ ]:
with open("label_encoder.pkl", "wb") as file:
    pickle.dump(encoder, file)

In [ ]:
print("Chatbot model trained successfully!")
print("Training examples:", len(sentences))
print("Number of intents:", len(set(labels)))

Chatbot model trained successfully!
Training examples: 120
Number of intents: 5


In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=30)

In [ ]:
test_accuracy = model.score(x_test,y_test)
print(test_accuracy)

1.0


In [ ]:
import pickle
import random
import json

# Load trained model
with open("chatbot_model.pkl", "rb") as f:
    model = pickle.load(f)

# Load vectorizer
with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

# Load label encoder
with open("label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

# Load the original intents data containing responses
# Corrected filename from 'ai_helpdesk_multilingual_v2.json' to 'chatbot_intent_augmentation.json'
with open("chatbot_intent_augmentation.json", "r", encoding="utf-8") as file:
    response_data_raw = json.load(file)

# The structure of the loaded data is a list of dictionaries, not a dictionary with an 'intents' key.
# We need to restructure it to match the expected format or adapt the chatbot_test function.
# Assuming the original data 'data' (from mDFjumB9BAEC) was also a list of dicts, and the intent extraction loop below expects response_data['intents']
# Let's create a dictionary with an 'intents' key from the loaded list to match the existing loop structure.
response_data = {"intents": []}
for item in response_data_raw:
    # Assuming each item in chatbot_intent_augmentation.json has 'text' and 'intent' keys.
    # To provide responses, we need to group by intent.
    # For simplicity, assuming 'chatbot_intent_augmentation.json' contains the actual intents data with 'tag' and 'responses' keys within 'intents' list.
    # If the format is different, this part will need further adjustment.
    # For now, let's assume the loaded JSON can be directly assigned, or the original 'data' object is what's needed.
    # Given the previous context `data = json.load(file)` for `chatbot_intent_augmentation.json`, it is a list of dictionaries.
    # The `chatbot_test` function expects `response_data["intents"]` to be a list where each item has a 'tag' and 'responses'.
    # The `chatbot_intent_augmentation.json` does not seem to have this structure directly.
    # The previous cell `mDFjumB9BAEC` loads `chatbot_intent_augmentation.json` into `data` variable. `data` is a list of dictionaries.
    # The `chatbot_test` function is written assuming `response_data` has a top-level key 'intents' which is a list of dictionaries, each with 'tag' and 'responses'.
    # Since the uploaded file 'chatbot_intent_augmentation.json' is a list of dictionaries, I need to restructure it.
    # I will adapt the loading to group responses by intent from the `chatbot_intent_augmentation.json`.
    pass # This section needs to be re-evaluated if the structure of 'chatbot_intent_augmentation.json' is a flat list.

# Re-loading the original data for responses, assuming it's structured like a typical intents.json file
# If 'chatbot_intent_augmentation.json' is a flat list of text/intent pairs, then we need to process it.
# Based on the earlier 'data' variable state, it's a list of dicts like `[{'text': 'what is the minimum attendance required?', 'intent': 'attendance_requirement'}, ...]`.
# To get responses, we need to create a mapping from intent tags to a list of responses.
response_map = {}
for item in response_data_raw:
    intent = item['intent']
    text = item['text'] # Assuming 'text' can also be a response, or we need a separate 'responses' field in the JSON.
    if intent not in response_map:
        response_map[intent] = []
    response_map[intent].append(text) # Using the training sentences as potential responses for now.

# Modify chatbot_test to use response_map

def chatbot_test(message):
    # Convert text into features
    X = vectorizer.transform([message])

    # Predict intent (numerical label)
    intent_numeric = model.predict(X)[0]

    # Convert numerical intent back to original tag
    intent_tag = encoder.inverse_transform([intent_numeric])[0]

    # Confidence
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)[0]
        confidence = max(probabilities)
    else:
        confidence = None

    # Get responses from dataset
    response = "I understood your query, but I don't have a response for it."

    if intent_tag in response_map:
        response = random.choice(response_map[intent_tag])

    print("User     :", message)
    print("Intent   :", intent_tag) # Print the actual tag

    if confidence is not None:
        print("Confidence:", round(confidence * 100, 2), "%")

    print("Bot      :", response)
    print("-" * 70)


# Test queries
test_queries = [
    "What is the minimum attendance required for exams?",
    "भाई परीक्षा देने के लिए कितनी attendance चाहिए?",
    "bhai meri attendance 68 hai exam de paunga kya",
    "meri attendance kam hai kya karu",
    "paper kab se start honge bhai",
    "When will the semester results be declared?",
    "result show nahi ho raha yaar",
    "scholarship ke liye kya documents chahiye",
    "college wifi connect nahi ho raha",
    "bhai exam form kaise bharu",
    "library mein book issue kaise karu",
    "hostel kaise milega",
    "placement ke liye kitna CGPA chahiye"
]

for query in test_queries:
    chatbot_test(query)


User     : What is the minimum attendance required for exams?
Intent   : attendance_requirement
Confidence: 60.49 %
Bot      : bhai exam ke liye kitni attendance required hai?
----------------------------------------------------------------------
User     : भाई परीक्षा देने के लिए कितनी attendance चाहिए?
Intent   : attendance_shortage
Confidence: 35.47 %
Bot      : low attendance ka kya karna hai?
----------------------------------------------------------------------
User     : bhai meri attendance 68 hai exam de paunga kya
Intent   : exam_eligibility
Confidence: 49.63 %
Bot      : can I appear for exams with low attendance?
----------------------------------------------------------------------
User     : meri attendance kam hai kya karu
Intent   : attendance_shortage
Confidence: 62.47 %
Bot      : attendance short hone par kya karna chahiye?
----------------------------------------------------------------------
User     : paper kab se start honge bhai
Intent   : result
Confidence: 33.

In [ ]:
import joblib

joblib.dump(model, "chatbot_v2_model.pkl")
joblib.dump(vectorizer, "chatbot_v2_vectorizer.pkl")

print("✅ Chatbot V2 model saved successfully!")

✅ Chatbot V2 model saved successfully!


In [ ]:
import joblib
joblib.dump(model, "chatbot_v2_model.pkl")
joblib.dump(vectorizer, "chatbot_v2_vectorizer.pkl")
joblib.dump(response_data_raw, "chatbot_v2_intents.pkl")
print("✅ Model, Vectorizer aur Intents save ho gaye!")

✅ Model, Vectorizer aur Intents save ho gaye!


In [ ]:
import joblib
model = joblib.load("chatbot_v2_model.pkl")
vectorizer = joblib.load("chatbot_v2_vectorizer.pkl")
intents = joblib.load("chatbot_v2_intents.pkl")
print("✅ Chatbot V2 successfully loaded!")

✅ Chatbot V2 successfully loaded!


In [ ]:
from google.colab import files

files.download("chatbot_v2_model.pkl")
files.download("chatbot_v2_vectorizer.pkl")
files.download("chatbot_v2_intents.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import joblib
from google.colab import files

joblib.dump(encoder, "chatbot_v2_label_encoder.pkl")
files.download("chatbot_v2_label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>